In [60]:
import pandas as pd
import numpy as np

In [61]:
import os
import librosa.display
import matplotlib.pyplot as plt
%matplotlib inline

In [62]:
from mutagen import File
import soundfile as sf

In [ ]:
root_folder = 'Old data that was not used to train the final model'

keys = ['Cm', 'C#m', 'Dm', 'D#m', 'Em', 'Fm', 'F#m', 'Gm', 'G#m', 'Am', 'A#m', 'Bm']

audio_files = []
for k in keys:
    key_folder_path = f"{root_folder}/{k}"
    for file in os.listdir(key_folder_path):
        if file.endswith(('.wav', '.mp3', '.ogg')):
            audio_files.append(f"{key_folder_path}/{file}")

durations = []
for file_path in audio_files:
    audio = File(file_path)
    if audio is not None and audio.info is not None:
        durations.append(audio.info.length * 1000) 

min_duration = min(durations)
print(f"Global shortest duration: {min_duration:.2f} ms")

Global shortest duration: 5825.29 ms


In [64]:
for k in keys:

    key_files = []
    
    key_folder_path = f"{root_folder}/{k}"
    
    for file in os.listdir(key_folder_path):
        if file.endswith(('.wav', '.mp3', '.ogg')):
            key_files.append(f"{key_folder_path}/{file}")

    key_output_path = f"{root_folder}/{k}_t"
    os.makedirs(key_output_path, exist_ok=True)

    name_inc = 0

    for file in key_files:

        save_path = f"{key_output_path}/{name_inc}.mp3"
        name_inc = name_inc + 1
        
        data, samplerate = sf.read(file)
        target_samples = int((min_duration / 1000) * samplerate)

        trimmed_data = data[:target_samples]

        # filename = os.path.basename(file)

        sf.write(save_path, trimmed_data, samplerate)

In [65]:
def create_spectrogram(audio_file, image_file):
    fig = plt.figure()
    ax = fig.add_subplot(1, 1, 1)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

    y, sr = librosa.load(audio_file)
    ms = librosa.feature.melspectrogram(y=y, sr=sr)
    log_ms = librosa.power_to_db(ms, ref=np.max)
    librosa.display.specshow(log_ms, sr=sr)

    fig.savefig(image_file)
    plt.close(fig)
    
def create_pngs_from_mp3s(input_path, output_path):
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    dir = os.listdir(input_path)

    for i, file in enumerate(dir):
        input_file = os.path.join(input_path, file)
        output_file = os.path.join(output_path, file.replace('.mp3', '.png'))
        create_spectrogram(input_file, output_file)

In [ ]:
for k in keys:
    
    input_path = 'Old data that was not used to train the final model'
    output_path = 'Old data that was not used to train the final model'
    create_pngs_from_mp3s(input_path, output_path)

In [67]:
from keras.preprocessing import image

def load_images_from_path(path, label):
    images = []
    labels = []

    for file in os.listdir(path):
        images.append(image.img_to_array(image.load_img(os.path.join(path, file), target_size=(224, 224, 3))))
        labels.append((label))
        
    return images, labels

def show_images(images):
    fig, axes = plt.subplots(1, 8, figsize=(20, 20), subplot_kw={'xticks': [], 'yticks': []})

    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i] / 255)
        
x = []
y = []

In [ ]:
label_counter = 0

for k in keys:
    
    images, labels = load_images_from_path('Old data that was not used to train the final model', label_counter)
    
    x += images
    y += labels

    label_counter = label_counter + 1

In [69]:
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y, test_size=0.3, random_state=0)

x_train_norm = np.array(x_train) / 255
x_test_norm = np.array(x_test) / 255

y_train_encoded = to_categorical(y_train)
y_test_encoded = to_categorical(y_test)

In [70]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Flatten, Dense

model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Flatten())
model.add(Dense(1024, activation='relu'))
model.add(Dense(12, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\LShel\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 109, 109, 128)  │        36,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 54, 54, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 52, 52, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1024)           │    18,875,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 12)             │        12,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,220,748 (73.32 MB)

 Trainable params: 19,220,748 (73.32 MB)

 Non-trainable params: 0 (0.00 B)

In [71]:
hist = model.fit(x_train_norm, y_train_encoded, validation_data=(x_test_norm, y_test_encoded), batch_size=10, epochs=20)

Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 616ms/step - accuracy: 0.0852 - loss: 3.3796 - val_accuracy: 0.0882 - val_loss: 2.4818
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 534ms/step - accuracy: 0.0370 - loss: 2.4867 - val_accuracy: 0.0882 - val_loss: 2.4854
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 529ms/step - accuracy: 0.0596 - loss: 2.5002 - val_accuracy: 0.0588 - val_loss: 2.4784
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 521ms/step - accuracy: 0.1467 - loss: 2.4736 - val_accuracy: 0.1176 - val_loss: 2.4788
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 527ms/step - accuracy: 0.1996 - loss: 2.4755 - val_accuracy: 0.0588 - val_loss: 2.4765
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 532ms/step - accuracy: 0.2014 - loss: 2.4429 - val_accuracy: 0.0882 - val_loss: 2.4735
Epoch 7/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 543ms/step - accuracy: 0.1521 - loss: 2.4288 - val_accuracy: 0.0882 - val_loss: 2.4700
Epoch 8/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 543ms/step - accuracy: 0.1753 - loss: 2.4349 - val_accuracy: 0.1176 - val_loss

In [72]:
# ----- #

In [ ]:
ex_path = 'Old data that was not used to train the final model'

In [102]:
d, s = sf.read(ex_path)

In [ ]:
t_s = int((min_duration / 1000) * s)

t_d = d[:t_s]

s_p = 'Old data that was not used to train the final model'

sf.write(s_p, t_d, s)

In [ ]:
i_p = 'Old data that was not used to train the final model'
o_p = 'Old data that was not used to train the final model'
create_pngs_from_mp3s(i_p, o_p)

In [105]:
def load_single_image(file_path):
    img = image.load_img(file_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    return img_array

In [ ]:
i = load_single_image('Old data that was not used to train the final model')

In [107]:
z = [i]

In [108]:
z_norm = np.array(z) / 255

In [109]:
predictions = model.predict(z_norm)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step


In [110]:
print(predictions)

[[7.7345280e-04 9.9855395e-05 9.8130453e-01 1.8290516e-03 1.0474433e-02
  1.4925766e-03 1.5335529e-03 2.3059433e-06 4.1218823e-06 6.9618860e-04
  1.6671749e-03 1.2283304e-04]]


In [111]:
predicted_class = np.argmax(predictions, axis=1)
print(f"Predicted class: {keys[0]}")

Predicted class: Cm
